# NDT7 (M-Lab) Data Prep — Singapore Broadband + Mobile, Province x Quarter

Aggregates `../../../data/ndt7/sg/mlab_sg_clean.parquet` (36.1M raw NDT7 test records, already ISP-classified and province-joined via
per-IP lookup + point-in-polygon) into province x quarter format, split into Broadband and
Mobile/Cellular parts, mirroring the same structure across all three NDT7 "tigger" countries
(Cambodia/Singapore/Vietnam).

**Rebuilt to use DuckDB instead of a manual pyarrow-batch-streaming loop** — DuckDB reads the
parquet file directly and does the tile-binning + GROUP BY aggregation out-of-core (no manual
batching code needed, no risk of the memory issues the streaming version was written to avoid).
The tile-binning and weighted-aggregation formulas are byte-for-byte unchanged from the pandas
version — verified against the prior pandas-based export (float-precision-only differences,
~1e-13, from AVG() accumulation order).

No province-name mapping needed — Singapore's raw `province` values already match `singapore_reference.csv`.

Same tile scheme as Ookla's own published tiles (zoom-16 slippy tiles, ~610m) — keeps
`n_tiles`/`is_reliable` comparable across Ookla and NDT7, and across countries:
`total_tests >= 100` only (n_tiles dropped for NDT7 — MaxMind gives city-centroid coordinates, so n_tiles measures cities-per-province, not data spread; Ookla keeps both).

**Outputs:**
- `data/exports/ndt7_singapore_province_quarterly.csv` — Broadband
- `data/exports/ndt7_mobile_singapore_province_quarterly.csv` — Mobile/Cellular
  (renamed from `ndt7_singapore_mobile_...` to match Ookla's `ookla_mobile_<country>_...`
  naming convention — position of "mobile" now matches across both pipelines)

In [1]:
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../../data/ndt7/sg/mlab_sg_clean.parquet'
SG_REF_CSV = '../../../data/reference/singapore_reference.csv'

ZOOM = 16
N_TILES = 2 ** ZOOM
MIN_TILE_TESTS = 3

### 1. Tile-Binning + Province-Quarter Aggregation (DuckDB)

All heavy row-level work (filtering, quarter-labeling, zoom-16 mercator tile assignment, GROUP BY tile x quarter x type x network_type) happens in one DuckDB SQL query against the raw parquet — no Python-side batching.

In [2]:
con = duckdb.connect()
con.execute("SET memory_limit='3GB'")                        # PH/ID ใหญ่ ต้องตั้ง
con.execute("SET temp_directory='../../../.tmp/duckdb'")   # ที่พักตอน spill
con.execute("SET preserve_insertion_order=false")

sql = f"""
WITH filtered AS (
    SELECT
        mean_throughput_mbps,
        min_rtt,
        type, network_type, province,
        year,
        CAST(CEIL(month / 3.0) AS INT) AS qtr,
        -- zoom-16 Web Mercator tile — ใช้เป็นคอลัมน์วินิจฉัยเท่านั้น ไม่ได้ใช้คิดค่าเฉลี่ย
        CAST(LEAST(FLOOR((longitude + 180) / 360 * 65536), 65535) AS BIGINT) AS tx,
        CAST(LEAST(FLOOR((1 - (ln(tan(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878)))
             + 1.0/cos(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878))))) / pi()) / 2 * 65536), 65535) AS BIGINT) AS ty
    FROM read_parquet('{RAW_PARQUET}')
    WHERE mean_throughput_mbps > 0
      AND province IS NOT NULL
      AND network_type IN ('broadband', 'cellular')
)
SELECT
    province, network_type, type,
    (CAST(year AS VARCHAR) || '-Q' || CAST(qtr AS VARCHAR)) AS year_q,
    AVG(mean_throughput_mbps)                       AS avg_thr,
    AVG(CASE WHEN min_rtt < 2000 THEN min_rtt END)  AS avg_lat,
    COUNT(*)                                        AS test_count,
    COUNT(DISTINCT tx * 65536 + ty) AS n_tiles
FROM filtered
GROUP BY province, network_type, type, year_q
"""

tile_agg_all = con.execute(sql).df()
print(f"province x quarter x type x network rows: {len(tile_agg_all):,}")
print(f"quarters: {len(tile_agg_all['year_q'].unique())} | province: {tile_agg_all['province'].nunique()}")
print(tile_agg_all.groupby('network_type')['test_count'].sum().apply(lambda x: f'{x:,}'))

province x quarter x type x network rows: 203
quarters: 12 | province: 5
network_type
broadband    17,640,552
cellular      3,822,288
Name: test_count, dtype: object


### 3. Province-Level Weighted Aggregation (per network type)

In [3]:
def build_province_quarterly(tile_agg_all, network_type, ref):
    d = tile_agg_all[tile_agg_all['network_type'] == network_type]
    print(f"[{network_type}] province x quarter x type rows: {len(d):,}")

    dl = d[d['type'] == 'download'].rename(columns={
        'avg_thr': 'avg_d_mbps', 'avg_lat': 'avg_lat_ms_wt', 'test_count': 'total_tests'})
    ul = d[d['type'] == 'upload'].rename(columns={'avg_thr': 'avg_u_mbps'})

    dl_stats = dl[['year_q', 'province', 'avg_d_mbps', 'avg_lat_ms_wt', 'total_tests', 'n_tiles']]
    ul_stats = ul[['year_q', 'province', 'avg_u_mbps']]

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
    master = master.rename(columns={'year_q': 'quarter'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)

    # NDT7 ใช้ total_tests อย่างเดียว ไม่ใช้ n_tiles เป็นเกณฑ์ (Ookla ยังใช้ทั้งคู่)
    # เหตุผล: NDT7 ได้พิกัดจาก MaxMind ซึ่งเป็น city centroid ทุก test ในเมืองเดียวกันจึงตกลง tile
    # เดียวกัน n_tiles จึงวัด "จังหวัดนี้มีกี่เมืองใน MaxMind" ไม่ได้วัดการกระจายตัวของข้อมูล
    # (ลาวทั้งประเทศมีพิกัดต่างกัน 33 จุด n_tiles สูงสุด = 3 -> เกณฑ์ >=5 เป็นไปไม่ได้)
    # คอลัมน์ n_tiles ยังเก็บไว้ให้ดูใน "Data Quality" ของ EDA
    master['is_reliable'] = master['total_tests'] >= 100
    print(f"[{network_type}] province x quarter rows: {len(master)} | "
          f"reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

In [4]:
ref = pd.read_csv(SG_REF_CSV)

---
## Part 1 — Broadband

In [5]:
broadband_master = build_province_quarterly(tile_agg_all, 'broadband', ref)
broadband_master.head()

[broadband] province x quarter x type rows: 109
[broadband] province x quarter rows: 55 | reliable: 52 (94.5%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Central Region,133.261775,43.173691,938305,1,84.603623,2023,1,True,Central Region,2,975310,107758,7304,132617.35,4241102.85
1,2023-Q1,East Region,155.648427,23.797620,6225,1,106.810077,2023,1,True,East Region,3,710690,107758,6368,132617.35,4241102.85
2,2023-Q1,North Region,44.236379,63.015331,55484,3,26.000492,2023,1,True,North Region,4,599400,107758,4456,132617.35,4241102.85
3,2023-Q1,North-East Region,3.615243,37.614000,1,1,0.088350,2023,1,False,North-East Region,1,974650,107758,8778,132617.35,4241102.85
4,2023-Q1,West Region,739.161352,1.854000,1,1,NaN,2023,1,False,West Region,4,944480,107758,4020,132617.35,4241102.85


In [6]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../../data/exports/ndt7_singapore_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)

Exported 55 rows -> ../../../data/exports/ndt7_singapore_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Central Region,2023-Q1,2023,1,133.261775,84.603623,43.173691,938305,1,True,Central Region,2,975310,107758,7304,132617.35,4241102.85
1,East Region,2023-Q1,2023,1,155.648427,106.810077,23.797620,6225,1,True,East Region,3,710690,107758,6368,132617.35,4241102.85
2,North Region,2023-Q1,2023,1,44.236379,26.000492,63.015331,55484,3,True,North Region,4,599400,107758,4456,132617.35,4241102.85


---
## Part 2 — Mobile/Cellular

In [7]:
mobile_master = build_province_quarterly(tile_agg_all, 'cellular', ref)
mobile_master.head()

[cellular] province x quarter x type rows: 94
[cellular] province x quarter rows: 47 | reliable: 45 (95.7%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Central Region,54.910556,72.780442,130743,1,11.488535,2023,1,True,Central Region,2,975310,107758,7304,132617.35,4241102.85
1,2023-Q1,East Region,241.178312,13.732857,14,1,8.241890,2023,1,False,East Region,3,710690,107758,6368,132617.35,4241102.85
2,2023-Q1,North Region,170.628223,50.747458,3741,2,21.708648,2023,1,True,North Region,4,599400,107758,4456,132617.35,4241102.85
3,2023-Q2,Central Region,58.792392,73.296919,125240,1,12.166691,2023,2,True,Central Region,2,975310,107758,7304,132617.35,4241102.85
4,2023-Q2,North Region,85.058164,64.585161,2617,1,15.605057,2023,2,True,North Region,4,599400,107758,4456,132617.35,4241102.85


In [8]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../../data/exports/ndt7_mobile_singapore_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)

Exported 47 rows -> ../../../data/exports/ndt7_mobile_singapore_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Central Region,2023-Q1,2023,1,54.910556,11.488535,72.780442,130743,1,True,Central Region,2,975310,107758,7304,132617.35,4241102.85
1,East Region,2023-Q1,2023,1,241.178312,8.241890,13.732857,14,1,False,East Region,3,710690,107758,6368,132617.35,4241102.85
2,North Region,2023-Q1,2023,1,170.628223,21.708648,50.747458,3741,2,True,North Region,4,599400,107758,4456,132617.35,4241102.85


## Summary

- Input: Singapore NDT7 raw test records, already province-joined + ISP-classified
- Output: province x quarter aggregates for Broadband and Mobile separately, tile-binned at
  Ookla's zoom-16 resolution, same `is_reliable` threshold as every Ookla country notebook and
  the other NDT7 "tigger" prep notebooks
- Engine: DuckDB (was: manual pyarrow-batch-streaming loop in pandas) — verified to reproduce
  the prior pandas-based export exactly (float-precision-only differences)